# BS_Agent_DingDan EDA 与清洗规则沉淀

This notebook is orchestration-only. Function code lives in `src/alg/cleaning/bs_agent_dingdan.py`.

## 0. 环境与路径配置

Configure read mode and paths. Modes: `sql_sample`, `sql_full_to_parquet`, `parquet`.

In [21]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from alg.cleaning.bs_agent_dingdan import (
    analyze_drug_code_consistency,
    analyze_enterprise,
    analyze_hospital_level,
    analyze_identifiers,
    analyze_numeric_desensitization,
    analyze_return_void,
    analyze_status,
    build_clean_table,
    # build_clean_model_audit_v2,
    build_column_maps,
    build_drug_category_map,
    build_ownership_map,
    build_paths,
    build_quality_report,
    build_region_mapping,
    ensure_output_dirs,
    load_env,
    load_input_dataframe,
    load_yaml,
    save_basic_profile,
    save_clean_outputs,
    save_source_order_profiles,
    save_quality_report,
    # save_v2_outputs,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

mode = "sql_sample"  # sql_sample | sql_full_to_parquet | parquet
max_rows = 5000
chunksize = 50000

paths = build_paths(PROJECT_ROOT)
ensure_output_dirs(paths)
schema = load_yaml(paths.config_path)
status_map = load_yaml(paths.status_map_path)
hospital_level_map = load_yaml(paths.hospital_level_map_path)
raw_to_alias, alias_to_raw, raw_columns = build_column_maps(schema)
sql_database_url, sql_table = load_env(paths.project_root)

print(f"project_root={paths.project_root}")
print(f"mode={mode}, max_rows={max_rows}, table={sql_table}")
print("SQL_DATABASE_URL configured:", bool(sql_database_url))

project_root=C:\Users\admin\Myprojects\for_git\algo_main
mode=sql_sample, max_rows=5000, table=BS_Agent_DingDan
SQL_DATABASE_URL configured: True


## 读取数据

CSV is only a small-sample fallback. Full local cache should be Parquet.

In [22]:
df_raw = load_input_dataframe(
    mode=mode,
    paths=paths,
    sql_database_url=sql_database_url,
    sql_table=sql_table,
    raw_columns=raw_columns,
    max_rows=max_rows,
    chunksize=chunksize,
)
df_raw.head()

,数据唯一标识符,订单明细ID,数据来源,订单名称,省,省编码,市,市编码,县区,县区编码,采购时间,药品编码,药品医保编码,通用名,商品名,剂型,规格,转换系数,采购单位,材质,采购价(元),采购数量,采购金额(元),配送数量,配送金额(元),到货数量,到货金额(元),采购地址,订单状态,企业编码,医疗机构等级,医疗机构详细等级,医疗机构编码,医疗机构,配送企业编码,配送企业,生产企业编码,生产企业,药品类别,数据更新时间,配送时间,到货时间,项目名称,所有制形式,退回数量,作废数量
0,996BF6CF-391B-4C5A-A528-4C5229A34A59,52470792591376bbfbcb76d777622338,安徽省基层药品交易系统,蚌埠市固镇县刘集镇中心卫生院20120805075736采购单,安徽省,340000,蚌埠市,340300,固镇县,340323,2012-08-05 00:00:00,4b98de3c6d633e40a667e553874075f0,XJ01XXL096B001040203712,注射用磷霉素钠,NaN,注射剂,按C₃H₇O₄P计2.0g(200万单位),1,支,钠钙玻璃模制注射剂瓶装,4.92,300.0,1475.0,0.0,0.0,0.0,0.0,NaN,已确认,207,二级,二级,YL341870,固镇县刘集镇中心卫生院,YLQY001064,安徽省恒济医药有限公司,F9BDA11C2EE443F49615B9AD6AFBD4AA,哈药集团制药总厂,全身用抗感染药/全身用抗菌药/其他抗菌药/其他抗菌药,2026-01-08 17:31:06.957,NaT,NaT,NaN,公立,NaN,0
1,67E98737-4563-4F18-84B1-E75DF2E8D5D0,CA1B62EB-B61B-4E8C-8E5C-3F2559EB8135,四川药品(疫苗)集中采购交易系统,成都市第二人民医院201508262141采购单,四川省,510000,成都市,510100,锦江区,510104,2015-08-26 23:07:00,XC08CAL013A025010403705,XC08CAL013A025010403705,拉西地平片,司乐平,素片,4mg,30,片,铝塑包装,63.33,510.0,32300.0,510.0,32300.0,510.0,32300.0,NaN,入库,207,三级,三级甲等,YL514812,成都市第二人民医院,YLQY022489,国药控股四川医药股份有限公司,E50FC20840554CF79ACAEF2582AE63BD,哈药集团三精明水药业有限公司,心血管系统/钙通道阻滞剂/主要作用于血管的选择性钙通道阻滞剂/二氢吡啶衍生物类,2025-04-06 11:17:25.737,NaT,NaT,NaN,公立,NaN,0
2,6E6555D6-F5F0-46DA-9112-BD01CB1788AC,98ad0ae7163cc6bcf665a441ad6aa68a,安徽省基层药品交易系统,六安市寿县张李乡卫生院20121105154213采购单,安徽省,340000,淮南市,340400,寿县,340422,2012-11-05 00:00:00,XJ01CAA040E001020303712,XJ01CAA040E001020303712,阿莫西林胶囊,益萨林,硬胶囊,0.5g,20,粒,铝塑,6.95,300.0,2085.0,0.0,0.0,0.0,0.0,NaN,已确认,207,未定级,一级丙等,YL344077,寿县张李乡卫生院,YLQY012060,国药控股六安有限公司,F9BDA11C2EE443F49615B9AD6AFBD4AA,哈药集团制药总厂,全身用抗感染药/全身用抗菌药/β-内酰胺类抗菌药，青霉素类/广谱青霉素类,2026-06-10 18:30:01.063,NaT,NaT,NaN,公立,NaN,0
3,A93EBD95-D30D-441D-BBD3-B61515FB4051,92c341a7caaf7044eae75eae9d0a085b,安徽省基层药品交易系统,宿州市埇桥区西二铺红十字卫生院20121209093445采购单,安徽省,340000,宿州市,341300,埇桥区,341302,2012-12-09 00:00:00,XJ01CAA040E001020303712,XJ01CAA040E001020303712,阿莫西林胶囊,益萨林,硬胶囊,0.5g,20,粒,铝塑,6.95,150.0,1042.5,0.0,0.0,0.0,0.0,NaN,已确认,207,未定级,一级,YL342477,宿州市埇桥区西二铺镇卫生院,YLQY004784,安徽华康医药集团有限公司,F9BDA11C2EE443F49615B9AD6AFBD4AA,哈药集团制药总厂,全身用抗感染药/全身用抗菌药/β-内酰胺类抗菌药，青霉素类/广谱青霉素类,2026-06-10 18:30:01.063,NaT,NaT,NaN,公立,NaN,0
4,EBB48B80-0501-47FD-8BB2-05E8C6962E96,147e56044653c7d08af73ccd1c6487d6,安徽省基层药品交易系统,亳州市涡阳县青町镇卫生院王小庙20121016210653采购单,安徽省,340000,亳州市,341600,涡阳县,341621,2012-10-16 00:00:00,XJ01CAA040E001020303712,XJ01CAA040E001020303712,阿莫西林胶囊,益萨林,硬胶囊,0.5g,20,粒,铝塑,6.95,30.0,208.5,0.0,0.0,0.0,0.0,NaN,已确认,207,未定级,一级甲等,YL344150,涡阳县青町镇卫生院,YLQY000164,安徽省亳州市药品采购供应站,F9BDA11C2EE443F49615B9AD6AFBD4AA,哈药集团制药总厂,全身用抗感染药/全身用抗菌药/β-内酰胺类抗菌药，青霉素类/广谱青霉素类,2026-06-10 18:30:01.063,NaT,NaT,NaN,公立,NaN,0


## 1. 基础规模检查

In [23]:
basic_summary, field_counts = save_basic_profile(df_raw, alias_to_raw, paths.export_eda)
basic_summary, field_counts.head(20)

({'row_count': 5000,
  'column_count': 46,
  'purchase_time_min': '2010-01-27 00:00:00',
  'purchase_time_max': '2024-12-30 15:03:37'},
       column  null_count  null_rate  distinct_count
 29      企业编码           0        0.0               4
 45      作废数量           0        0.0               1
 25      到货数量           0        0.0             129
 26   到货金额(元)           0        0.0             819
 15        剂型           0        0.0              12
 33      医疗机构           0        0.0            3429
 30    医疗机构等级           0        0.0               3
 32    医疗机构编码           0        0.0            3429
 31  医疗机构详细等级           0        0.0              14
 8         县区           0        0.0            1267
 9       县区编码           0        0.0            1281
 6          市           0        0.0             309
 7        市编码           0        0.0             309
 43     所有制形式           0        0.0               3
 0    数据唯一标识符           0        0.0            5000
 39    数据更新时间   

## 2. 唯一标识符检查

In [24]:
id_report, duplicate_samples = analyze_identifiers(df_raw, alias_to_raw, paths.export_eda)
id_report, duplicate_samples.head()

(                      check value
 0            row_uid_unique  True
 1    order_detail_id_unique  True
 2          row_uid_distinct  5000
 3  order_detail_id_distinct  5000,
 Empty DataFrame
 Columns: [订单明细ID, 数据唯一标识符, 订单状态, 数据更新时间, 配送企业, 采购数量, 配送数量, 到货数量]
 Index: [])

## 3. 数据来源与订单名称

In [ ]:
source_top, order_name_top = save_source_order_profiles(df_raw, alias_to_raw, paths.export_eda)
source_top.head(), order_name_top.head()

## 4. 地区字段清洗

In [ ]:
region_map, region_conflicts = build_region_mapping(df_raw, alias_to_raw, paths.export_eda, paths.export_mappings)
region_map.head(), region_conflicts.head()

## 5. 药品编码与医保编码一致性

In [ ]:
drug_code_report = analyze_drug_code_consistency(df_raw, alias_to_raw, paths.export_eda)
drug_code_report

## 6. 数值字段脱敏影响检查

Updated policy: quantity fields share multiplier q; amount fields share multiplier m. `delivery_rate`, `arrival_rate`, `overall_arrival_rate`, quantity trends, amount trends, and order frequency trends are allowed. `price_from_amount_quantity` is forbidden: do not infer real unit price from amount / quantity.

In [ ]:
numeric_report = analyze_numeric_desensitization(df_raw, alias_to_raw, paths.export_eda)
numeric_report

## 7. 订单状态枚举与初步归类

In [ ]:
status_counts, status_review, status_unmapped = analyze_status(
    df_raw, alias_to_raw, status_map, paths.export_eda, paths.export_mappings
)
status_counts.head(), status_review.head(), status_unmapped.head()

## 8. 医疗机构等级清洗

In [ ]:
hospital_counts, hospital_review, hospital_unmapped = analyze_hospital_level(
    df_raw, alias_to_raw, hospital_level_map, paths.export_eda, paths.export_mappings
)
hospital_counts.head(), hospital_review.head(), hospital_unmapped.head()

## 9. 企业字段关系检查

In [ ]:
enterprise_report = analyze_enterprise(df_raw, alias_to_raw, paths.export_eda)
enterprise_report

## 10. 药品类别编码

In [ ]:
drug_category_counts = build_drug_category_map(df_raw, alias_to_raw, paths.export_mappings)
drug_category_counts.head()

## 11. 所有制形式编码

In [ ]:
ownership_map = build_ownership_map(df_raw, alias_to_raw, paths.export_mappings)
ownership_map.head()

## 12. 退回数量与作废数量

In [ ]:
return_void_report = analyze_return_void(df_raw, alias_to_raw, paths.export_eda)
return_void_report

## 13. 生成 clean 表

In [ ]:
df_clean = build_clean_table(
    df_raw,
    schema=schema,
    raw_to_alias=raw_to_alias,
    status_map=status_map,
    hospital_level_map=hospital_level_map,
    drug_category_counts=drug_category_counts,
)
save_clean_outputs(df_clean, paths)
df_clean.head()

## 14. 生成第二轮 clean/model/audit 样本输出

In [ ]:
df_clean_v2, df_model_sample, df_audit_sample, numeric_report_v2, profile_v2 = save_v2_outputs(
    df_raw,
    paths,
    schema=schema,
    raw_to_alias=raw_to_alias,
    hospital_level_map=hospital_level_map,
    drug_category_counts=drug_category_counts,
)
df_clean_v2.head(), df_model_sample.head(), df_audit_sample.head()

## 15. 生成 Markdown 数据质量报告

In [ ]:
quality_report = build_quality_report(
    schema=schema,
    basic=basic_summary,
    status_review=status_review,
    status_unmapped=status_unmapped,
    hospital_review=hospital_review,
    hospital_unmapped=hospital_unmapped,
)
report_path = save_quality_report(quality_report, paths.export_eda)
print(report_path)
print(quality_report[:1000])
print(profile_v2[:1000])